In [2]:
import os, requests
from dotenv import load_dotenv

load_dotenv()
HEADERS = {
    "Accept": "application/vnd.github+json",
    "Authorization": f"Bearer {os.environ['GITHUB_TOKEN']}",
    "X-GitHub-Api-Version": "2026-03-10",
}
url = "https://api.github.com/repos/microsoft/PowerToys/issues"
params = {"state": "all", "sort": "updated", "direction": "asc",
          "since": "2026-10-01T00:00:00Z", "per_page": 100}

r = requests.get(url, headers=HEADERS, params=params, timeout=30)
r.raise_for_status()
issues = r.json()
print(len(issues), "rânduri; cereri rămase:", r.headers["x-ratelimit-remaining"])
print("pagina următoare:", r.links.get("next", {}).get("url"))
print(sum("pull_request" in i for i in issues), "PR-uri în pagină")

100 rânduri; cereri rămase: 4944
pagina următoare: https://api.github.com/repositories/184456251/issues?state=all&sort=updated&direction=asc&since=2026-10-01T00%3A00%3A00Z&per_page=100&after=Y3Vyc29yOnYyOpLPAAABoQQYjADOQ4OfYg%3D%3D&page=2
34 PR-uri în pagină


In [2]:
import json, pathlib

pathlib.Path("../data").mkdir(exist_ok=True)
with open("../data/sample_issues.json", "w", encoding="utf-8") as f:
    json.dump(issues, f, ensure_ascii=False, indent=2)

In [3]:
first = issues[0]
print(sorted(first.keys()))
print({k: first[k] for k in ["id", "number", "title", "state", "state_reason",
                             "created_at", "updated_at", "closed_at"]})
print(first["user"]["login"], [l["name"] for l in first["labels"]])

['active_lock_reason', 'assignees', 'author_association', 'body', 'closed_at', 'closed_by', 'comments', 'comments_url', 'created_at', 'events_url', 'html_url', 'id', 'issue_dependencies_summary', 'issue_field_values', 'labels', 'labels_url', 'locked', 'milestone', 'node_id', 'number', 'performed_via_github_app', 'pinned_comment', 'reactions', 'repository_url', 'state', 'state_reason', 'sub_issues_summary', 'timeline_url', 'title', 'type', 'updated_at', 'url', 'user']
{'id': 4558840702, 'number': 48234, 'title': 'FancyZones “Move newly created windows to their last known zone” causes Adobe Illustrator canvas to open blank on multi-monitor setup', 'state': 'open', 'state_reason': None, 'created_at': '2026-05-31T20:10:17Z', 'updated_at': '2026-10-01T01:11:35Z', 'closed_at': None}
majd3lawneh ['Issue-Bug', 'Product-FancyZones', 'Needs-Triage']


In [4]:
def fetch_all(url, params):
    rows, pages = [], 0
    while url:
        r = requests.get(url, headers=HEADERS, params=params, timeout=30)
        r.raise_for_status()
        rows.extend(r.json())
        pages += 1
        url = r.links.get("next", {}).get("url")
        params = None  # linkul next conține deja parametrii
    return rows, pages

since = "2026-09-30T00:00:00Z"
rows, pages = fetch_all(url, {"state": "all", "sort": "updated", "direction": "asc",
                              "since": since, "per_page": 100})
print(len(rows), "rânduri în", pages, "pagini")
print("toate după since:", all(r["updated_at"] >= since for r in rows))

499 rânduri în 5 pagini
toate după since: True


In [5]:
from urllib.parse import urlparse, parse_qs

def page_count(repo):
    r = requests.get(f"https://api.github.com/repos/{repo}/issues", headers=HEADERS,
                     params={"state": "all", "per_page": 100}, timeout=30)
    r.raise_for_status()
    last = r.links.get("last", {}).get("url")
    return int(parse_qs(urlparse(last).query)["page"][0]) if last else 1

repos = ["googleapis/python-bigquery", "pandas-dev/pandas", "microsoft/PowerToys"]
counts = {repo: page_count(repo) for repo in repos}
print(counts, "| total cereri:", sum(counts.values()))

{'googleapis/python-bigquery': 1, 'pandas-dev/pandas': 1, 'microsoft/PowerToys': 1} | total cereri: 3


1. Un issue modificat de trei ori între două rulări: de câte ori îl primești și cu ce stare?
2. De ce nu e suficient să ceri exact de la ultimul watermark? Gândește-te la o rulare care pică la jumătate.
3. Ce înseamnă state_reason = not_planned pentru metrica de SLA?

In [3]:
def check(repo):
    info = requests.get(f"https://api.github.com/repos/{repo}", headers=HEADERS, timeout=30).json()
    r = requests.get(f"https://api.github.com/repos/{repo}/issues", headers=HEADERS, timeout=30,
                     params={"state": "all", "sort": "updated", "direction": "desc", "per_page": 1})
    print(repo, "| arhivat:", info["archived"], "| ultima activitate:", r.json()[0]["updated_at"])

for repo in ["streamlit/streamlit", "astral-sh/uv", "pandas-dev/pandas", "microsoft/PowerToys"]:
    check(repo)

streamlit/streamlit | arhivat: False | ultima activitate: 2026-10-08T11:38:33Z
astral-sh/uv | arhivat: False | ultima activitate: 2026-10-08T11:32:19Z
pandas-dev/pandas | arhivat: False | ultima activitate: 2026-10-08T11:21:48Z
microsoft/PowerToys | arhivat: False | ultima activitate: 2026-10-08T11:37:39Z
